# Dimension Interpretation using Gemini

This notebook interprets the meaning of the 20 PCA dimensions of video embeddings.
It queries the Gemini model to analyze the 20 highest and 20 lowest scoring videos for each dimension.
To ensure the definitions are mutually exclusive and collectively exhaustive, the prompt includes the definitions of previously analyzed dimensions.

## 1) Install dependencies and set up Environment

In [5]:
!pip install -q pandas google-generativeai

import pandas as pd
import ast
import json
import time
from pathlib import Path
try:
    from google.colab import ai
except ImportError:
    pass


try:
    from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print('Not running in Colab or missing genai library. Falling back to local execution and dummy definitions.')
    is_colab = False
except Exception as e:
    print(f"Error initializing Google Colab environment: {e}")
    is_colab = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Load Data and Prepare Embeddings

We load the canonical video embeddings and expand the 20D array into separate columns (`dim_0` to `dim_19`).
We provide a local dummy data fallback so the notebook can be tested headlessly without Drive access.

In [7]:
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

# Fallback mechanism for headless execution
if not DATA_PATH.exists():
    print(f"Warning: {DATA_PATH} not found. Creating a dummy dataset for testing.")

    # Create 100 dummy rows with random embeddings to allow the script to run
    import numpy as np
    dummy_data = []
    for i in range(100):
        dummy_data.append({
            'video_id': f'vid_{i}',
            'channel_name': f'Channel_{i % 5}',
            'video_title': f'Dummy Video Title {i}',
            'view_count': np.random.randint(1000, 100000),
            'embedding_20d': str([float(x) for x in np.random.randn(20)])
        })
    df = pd.DataFrame(dummy_data)
else:
    df = pd.read_csv(DATA_PATH)

dim_cols = []
if not df.empty:
    dim_cols = [c for c in df.columns if c.startswith('embedding_reduced_')]
    if len(dim_cols) != 20:
        if 'embedding_20d' in df.columns:
            import ast
            if isinstance(df['embedding_20d'].iloc[0], str):
                df['embedding_20d'] = df['embedding_20d'].apply(ast.literal_eval)
            dim_cols = [f'dim_{i}' for i in range(20)]
            emb_df = pd.DataFrame(df['embedding_20d'].tolist(), columns=dim_cols, index=df.index)
            df = pd.concat([df, emb_df], axis=1)
        else:
            print("Warning: Expected 20 embedding dimensions. Ensure the data format is correct.")

print(f"Loaded {len(df)} rows. Found {len(dim_cols)} dimension columns.")

Loaded 1344 rows. Found 20 dimension columns.


## 3) Interpretation Algorithm
We loop through each dimension. For each one, we:
1. Sort the dataset to find the top 20 and bottom 20 videos.
2. Build a prompt including those video titles and the previously generated definitions.
3. Query Gemini for the interpretation.

In [10]:
def call_gemini(prompt: str) -> str:
    """Calls the Gemini API if available, otherwise returns a dummy response."""
    if is_colab:
        try:
            # We add a small sleep to respect rate limits if calling in a loop
            time.sleep(2)
            print(prompt)
            response = ai.generate_text(prompt)
            print(response)
            return response
        except Exception as e:
            print(f"Error calling Gemini: {e}")
            return "Error generating definition."
    else:
        return "Dummy interpretation for headless testing."

interpretations = []
previous_definitions = ""

for dim_idx, dim_col in enumerate(dim_cols):
    if df.empty:
        break

    print(f"Processing {dim_col}...")

    # Sort and get top/bottom 20
    df_sorted = df.sort_values(by=dim_col, ascending=False)

    # Assuming video_title and channel_name are the main context columns
    title_col = 'video_title' if 'video_title' in df.columns else 'title'
    channel_col = 'channel_name' if 'channel_name' in df.columns else 'channelId'

    if title_col not in df.columns:
        # Fallback to whatever string column exists
        title_col = df.columns[0]

    top_20 = df_sorted.head(20)
    bottom_20 = df_sorted.tail(20)

    top_texts = []
    for _, row in top_20.iterrows():
        top_texts.append(f"- {row.get(title_col, 'Unknown')}")

    bottom_texts = []
    for _, row in bottom_20.iterrows():
        bottom_texts.append(f"- {row.get(title_col, 'Unknown')}")

    top_str = "\n".join(top_texts)
    bottom_str = "\n".join(bottom_texts)

    prompt = f"""
    You are an expert content analyst. We have extracted 20 semantic dimensions from a set of YouTube videos.
    Your task is to define what dimension {dim_idx} represents based on its highest and lowest scoring videos.

    Here are the top 20 videos that score the HIGHEST on dimension {dim_idx}:
    {top_str}

    Here are the bottom 20 videos that score the LOWEST on dimension {dim_idx}:
    {bottom_str}

    To ensure definitions are mutually exclusive and collectively exhaustive, here are the definitions of the PREVIOUSLY analyzed dimensions:
    {previous_definitions if previous_definitions else 'None yet.'}

    Based on the contrast between the highest and lowest videos, and keeping your definition distinctly different from the previously defined dimensions, provide a concise (1-2 sentences) and insightful definition for dimension {dim_idx}.
    Focus on the semantic meaning, content type, style, or topic that this dimension captures. Do not repeat previous definitions.
    """

    definition = call_gemini(prompt)
    print(f"Definition {dim_idx}: {definition}\n")

    # Store result
    dim_data = {
        "dimension_index": dim_idx,
        "dimension_name": dim_col,
        "definition": definition,
        "top_20_sample": top_texts[:5], # Store a small sample for the artifact to keep size manageable, or all 20
        "bottom_20_sample": bottom_texts[:5]
    }
    interpretations.append(dim_data)

    # Update previous definitions
    previous_definitions += f"\n- Dimension {dim_idx}: {definition}"


Processing embedding_reduced_01...

    You are an expert content analyst. We have extracted 20 semantic dimensions from a set of YouTube videos.
    Your task is to define what dimension 0 represents based on its highest and lowest scoring videos.

    Here are the top 20 videos that score the HIGHEST on dimension 0:
    - How to Get Your Customers to Stay FOREVER
- The Powerful Alternative To Fine-Tuning
- How to Earn an Income on Gold
- If You Have a Hard Life, Watch This
- Success IS NOT What you Think it Will Be
- My Life Advice for Teenagers
- The Time the United States Ran Out of Money
- The Time the United States Ran Out of Money
- Build a Mind So Strong It Scares People
- How to make progress faster than everyone
- #1 Secret to Taking Full Responsibility for Your Life
- How to Change Your Life in 30 Days
- Make Your Own Luck
- The UK is a Warning to the Rest of the World
- The 6 Levels of Making Money
- How To Get Your First Users
- The Reason I Studied 500 Years of History
- 

## 4) Simultaneous MECE Generation

We now perform a single API call to `gemini-1.5-pro` to generate all dimension definitions simultaneously. This allows the model to inherently ensure all definitions are Mutually Exclusive and Collectively Exhaustive (MECE) without iterating.


In [ ]:
import json

simultaneous_interpretations = []

if df.empty:
    print("Dataset empty, skipping simultaneous MECE generation.")
else:
    print("Preparing simultaneous MECE prompt...")
    # Build prompt structure
    all_dimensions_data = ""
    for dim_idx, dim_col in enumerate(dim_cols):
        df_sorted = df.sort_values(by=dim_col, ascending=False)
        top_20 = df_sorted.head(20)
        bottom_20 = df_sorted.tail(20)
        
        top_str = "\n".join([f"- {row.get(title_col, 'Unknown')}" for _, row in top_20.iterrows()])
        bottom_str = "\n".join([f"- {row.get(title_col, 'Unknown')}" for _, row in bottom_20.iterrows()])
        
        all_dimensions_data += f"""
        Dimension {dim_idx} ({dim_col}):
        TOP 20 VIDEOS:
        {top_str}
        
        BOTTOM 20 VIDEOS:
        {bottom_str}
        
        """

    simultaneous_prompt = f"""
    You are an expert content analyst. We have extracted 20 semantic dimensions from a set of YouTube videos.
    Your task is to define what EACH dimension represents based on its highest and lowest scoring videos.
    
    CRITICAL INSTRUCTION: Your definitions MUST be Mutually Exclusive and Collectively Exhaustive (MECE). 
    No two dimensions should have overlapping definitions. 
    
    Here is the data for all dimensions:
    {all_dimensions_data}
    
    Based on the contrast between the highest and lowest videos for each dimension, provide a concise (1-2 sentences) and insightful definition.
    Focus on the semantic meaning, content type, style, or topic that each dimension captures.
    
    You MUST return your response as a JSON array of objects. Each object must have exactly these keys:
    "dimension_index" (integer)
    "dimension_name" (string, e.g., "Dimension 0" or an insightful name)
    "top_description" (string, what the top videos represent)
    "bottom_description" (string, what the bottom videos represent)
    "definition" (string, the overall concise definition)
    """

    if is_colab:
        try:
            print("Calling gemini-1.5-pro for simultaneous generation...")
            # Use gemini-1.5-pro and enforce JSON response
            model = ai.GenerativeModel("gemini-1.5-pro", generation_config={"response_mime_type": "application/json"})
            response = model.generate_content(simultaneous_prompt)
            print("Received response.")
            simultaneous_interpretations = json.loads(response.text)
            
            # Map dim_col names to the result for consistency
            for item in simultaneous_interpretations:
                idx = item.get("dimension_index", 0)
                if idx < len(dim_cols):
                    item["dimension_column"] = dim_cols[idx]
                
        except Exception as e:
            print(f"Error calling Gemini for simultaneous generation: {e}")
            simultaneous_interpretations = [{"error": "Failed to generate simultaneous interpretations."}]
    else:
        print("Headless mode: Creating dummy simultaneous interpretations.")
        simultaneous_interpretations = []
        for i, dim_col in enumerate(dim_cols):
            simultaneous_interpretations.append({
                "dimension_index": i,
                "dimension_column": dim_col,
                "dimension_name": f"Dummy Name {i}",
                "top_description": "Dummy top description",
                "bottom_description": "Dummy bottom description",
                "definition": "Dummy simultaneous definition for headless testing."
            })


## 5) Export Artifacts
We save the generated definitions to a JSON file at the standard Graphiko analysis path.

In [4]:
output_root = Path('/content/drive/MyDrive/Graphiko/analysis/dimension_interpretation/')

if not is_colab or not output_root.exists():
    print("Using local directory for output fallback.")
    output_root = Path('./dimension_interpretation_output')

output_dir = output_root / 'latest'
output_dir.mkdir(parents=True, exist_ok=True)

artifact_path = output_dir / 'dimension_interpretations.json'

artifact = {
    "schema_version": "1.0.0",
    "artifacts": {
        "dimension_interpretations": interpretations,
        "simultaneous_interpretations": simultaneous_interpretations
    },
    "run_summary": {
        "models_used": ["gemini-1.5-flash", "gemini-1.5-pro"],
        "dimensions_processed": len(interpretations),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    }
}

with open(artifact_path, 'w') as f:
    json.dump(artifact, f, indent=2)

print(f"Exported artifact to {artifact_path}")


Using local directory for output fallback.
Exported artifact to dimension_interpretation_output/latest/dimension_interpretations.json
